<a href="https://colab.research.google.com/github/elenann-cpu/AI-Scanner-App/blob/master/Task_4_(part_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy scipy matplotlib scikit-learn sklearn-pandas seaborn

In [ ]:
!git clone https://github.com/nogrady/PPML.git

In [ ]:
import sys
sys.path.append('/content/PPML/Ch7 & Ch8')

In [ ]:
!ls "/content/PPML/Ch7 & Ch8"

In [ ]:
import os
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"

if not os.path.exists("bank-additional.zip"):
    !wget $url

if not os.path.exists("bank-additional"):
    !unzip -o bank-additional.zip

df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')

print(f"Dataset: ({df.shape[0]}, {df.shape[1]})")

df.head(5)

# **Data sanitisation on the Banking dataset**

# Suppresion

In [ ]:
df.drop(columns=["poutcome", "default", "pdays", "duration"], inplace=True)

display(df.head())

print("-" * 30)
print(f"Remaining columns: {len(df.columns)}")
print(f"Dataset: {df.shape}")

# Generalization

In [ ]:
df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')

age_bins = [0, 20, 30, 40, 50, 60, 70, 110]
age_labels = ['<=20', '21-30', '31-40', '41-50', '51-60', '61-70', '>70']
df['age_generalized'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

df['job'] = df['job'].astype(str).str.strip()

job_map = {
    'admin.': 'White-Collar',
    'management': 'White-Collar',
    'technician': 'White-Collar',
    'blue-collar': 'Blue-Collar',
    'services': 'Blue-Collar',
    'housemaid': 'Blue-Collar',
    'entrepreneur': 'Self-Employed',
    'self-employed': 'Self-Employed',
    'retired': 'Not-In-Labor-Force',
    'student': 'Not-In-Labor-Force',
    'unemployed': 'Not-In-Labor-Force',
    'unknown': 'Unknown'
}
df['job_generalized'] = df['job'].map(job_map)

df_generalized = df.drop(columns=['age', 'job'])

df = df_generalized.copy()

display(df[['age_generalized', 'job_generalized']].head())

# Perturbation

Listing 7.1 Perturbing age and race

In [ ]:
categorical = ['job']
continuous = ['age']

unchanged = []
for col in df.columns:
    if col not in categorical and col not in continuous:

        if not col.endswith('_perturbed'):
            unchanged.append(col)

best_distributions = []
for col in continuous:
    data = df[col]
    best_dist_name, best_dist_params = best_fit_distribution(data, 500)
    best_distributions.append((best_dist_name, best_dist_params))

print("Columns to perturb (categorical):", categorical)
print("Columns to perturb (continuous):", continuous)
print("Columns that remain unchanged:", len(unchanged))

Listing 7.2 Perturbation for both numerical and categorical values

In [ ]:
def perturb_data(df, unchanged_cols, categorical_cols, continuous_cols, best_distributions, n, seed=0):
    np.random.seed(seed)
    data = {}

    for col in categorical_cols:
        counts = df[col].value_counts()
        data[col] = np.random.choice(list(counts.index),
                                     p=(counts/len(df)).values, size=n)

    for col, bd in zip(continuous_cols, best_distributions):
        dist = getattr(scipy.stats, bd[0])
        data[col] = np.round(dist.rvs(size=n, *bd[1]))

    for col in unchanged_cols:
        data[col] = df[col].values[:n]

    return pd.DataFrame(data, columns=unchanged_cols + categorical_cols + continuous_cols)

gendf = perturb_data(df, unchanged, categorical, continuous, best_distributions, n=len(df))

gendf.head()

# Anatomization

In [ ]:
df_anatomization = df.copy()

df_anatomization['group_id'] = df_anatomization.groupby(['education', 'marital']).ngroup()

dataset_1_qi = df_anatomization[['group_id', 'education', 'marital']]

dataset_2_sensitive = df_anatomization[['group_id', 'housing']]

print("DATASET 1: Quasi-Identifiers (Education + Marital)")
display(dataset_1_qi.head())

print("\n" + "-"*50 + "\n")

print("DATASET 2: Sensitive Attributes (Housing Loan Status)")
display(dataset_2_sensitive.head())

# K-anonimity

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set(style="darkgrid")

class Protect:
    def __init__(self, df, k, quasi_ids):
        self.df = df.copy()
        self.k = k
        self.quasi_ids = quasi_ids

    def protect(self):
        df = self.df.copy()

        for col in self.quasi_ids:
            if df[col].dtype in ['int64', 'float64']:

                df[col] = pd.cut(df[col], bins=5, labels=False)
            else:
                df[col] = df[col].astype(str)

        result = df.groupby(self.quasi_ids).filter(lambda x: len(x) >= self.k)

        return result

df = gendf.copy()

display(df.head())

print("-" * 30)
print(f"Shape before k-anonymity: {df.shape}")

In [ ]:
quasi_ids = ['age', 'job']

prot = Protect(df, k=5, quasi_ids=quasi_ids)

print("Privacy model set to: k-anonymity")
print(f"K-value: {prot.k}")
print(f"Quasi-identifiers: {prot.quasi_ids}")

In [ ]:
itypes = pd.Series(data='insensitive', index=df.columns)

itypes['age'] = 'quasi'
itypes['job'] = 'quasi'

print(itypes)

In [ ]:
prot_df = prot.protect()

print(f"Dataset is now 5-anonymized (True Anonymization).")

display(pd.concat([prot_df.head(5), prot_df.tail(5)]))

print("-" * 30)
print(f"Shape after k-anonymity: {prot_df.shape}")

min_size = prot_df.groupby(prot.quasi_ids).size().min()

print(f"Smallest group size: {min_size} (Goal: >= {prot.k})")